# USD/JPY Donchian Breakout Research 

## 1. Objective & Data Setup

### 1.1 Hypothesis
USDJPY exhibits persistent time-series momentum that can be systematically exploited via Donchian Channel breakouts, net of FX spreads, swap carry, and session microstructure. 

**Inspiration:** <br>
Hurst, B., Ooi, Y. H., & Pedersen, L. H. (2013). Demystifying managed futures. Journal of Investment Management.
https://www.aqr.com/-/media/AQR/Documents/Insights/Journal-Article/Demystifying-Managed-Futures.pdf

### 1.2 In-Sample / Out-of-Sample Split
* **Maximum Date ($T_{\text{max}}$):** The latest timestamp available in the CSV.
* **Out-of-Sample Start ($T_{\text{OOS\_start}}$):** Computed as $T_{\text{max}} - 60 \text{ months}$ ($5 \text{ years}$).
* **In-Sample (IS):** Earliest record up to the day immediately preceding $T_{\text{OOS\_start}}$.
* **Out-of-Sample (OOS):** $T_{\text{OOS\_start}}$ to $T_{\text{max}}$.



## 2. Strategy

### 2.1. Signal Generation

The strategy uses a classical Donchian Channel. 

For a given lookback window ($N$ days), it calculates:
* **Upper Band:** The highest Close price over the previous $N-1$ days.
* **Lower Band:** The lowest Close price over the previous $N-1$ days.

> **Explanation:** Using `.shift(1)` to inspect historical days ensures zero look-ahead bias. Today's signal strictly uses price data up to yesterday's close.    

* **Long Signal ($1$):** Generated when today's Close breaks above the Upper Band.
* **Short Signal ($-1$):** Generated when today's Close breaks below the Lower Band.
* **Hold:** If neither band is broken, the strategy holds its previous position using forward fill (`df["signal"].ffill()`). The strategy is always active in the market (either fully long or fully short).

In [8]:
# src/strategy.py

def donchian_breakout(df: pd.DataFrame, lookback: int) -> pd.DataFrame:
    df["upper"] = df["close"].rolling(lookback - 1).max().shift(1)
    df["lower"] = df["close"].rolling(lookback - 1).min().shift(1)
    df["signal"] = np.nan
    df.loc[df["close"] > df["upper"], "signal"] = 1
    df.loc[df["close"] < df["lower"], "signal"] = -1
    df["signal"] = df["signal"].ffill()
    return df

### 2.2. Execution, Costs & Returns

Once a signal is generated at the close of day $t$ (the 5:00 PM New York rollover), the trade is executed at that close price, and P&L is realised over the subsequent interval ($t \to t+1$).

#### Components Framework

1. **Strategy Return:**
   $$\text{strategy\_return}_t = \text{signal}_t \times \ln\left(\frac{\text{Close}_{t+1}}{\text{Close}_t}\right)$$

   * **If Long ($\text{signal}_t = 1$):** The strategy earns the asset's daily gain.
   * **If Short ($\text{signal}_t = -1$):** The strategy earns the inverse of the daily return (profiting when prices fall).

   > **Explanation:** Shifting returns to the next day (`.shift(-1)`) guarantees that a signal established on Monday's close captures the actual price movement from Monday's close to Tuesday's close.

2. **Pip-Based Transaction Costs:**
    $$C_t = \ln\left(1 - \frac{0.005}{\text{Close}_t}\right) \times 2$$

    > **Explanation:** $0.005$ represents the absolute JPY value of a $0.5\text{-pip}$ spread ($0.5 \text{ pips} \times 0.01 \text{ JPY/pip}$), and $\times 2$ represents the absolute change in position required for a full position flip (e.g., closing long & opening short).

3. **Carry Adjustments:**
    $$\text{Carry}_t = \text{signal}_t \times \frac{\ln(1 + \text{annual\_carry})}{252\text{ trading days}}$$

    > **Explanation:** Because the strategy holds positions overnight indefinitely, it is subject to the USD/JPY interest rate differential. The model applies a daily carry adjustment ($2\%$ annualised proxy). 


#### Net Daily Return

Combining gross strategy returns, transaction costs, and overnight carry adjustments:

$$\text{Net Return}_t = \text{strategy\_return}_t - C_t + \text{Carry}_t$$

In [ ]:
# src/backtest.py

def signal_returns(price_data: pd.DataFrame, pip_cost: float = 0.5, point: float = 0.01, annual_carry: float = 0.02) -> pd.DataFrame:
    # 1. Strategy return from 5PM NY rollover
    price_data["daily_log_return"] = np.log(price_data["close"]).diff().shift(-1)
    price_data["daily_strategy_return"] = price_data["signal"] * price_data["daily_log_return"]

    # 2. FX Transaction Cost
    pip_cost_log_percentage = np.log(1 - (pip_cost * point) / price_data["close"])
    absolute_position_change = price_data["signal"].diff().abs()
    price_data["daily_strategy_return"] -= absolute_position_change * pip_cost_log_percentage

    # 3. Carry Adjustment
    daily_carry_log_percentage = np.log(1 + annual_carry) / 252
    price_data["daily_strategy_return"] += price_data["signal"] * daily_carry_log_percentage

    # Final Net Equity Curve
    price_data["cumulative_equity_curve"] = price_data["daily_strategy_return"].cumsum()
    return price_data

## 3. Optimisation Framework
The strategy optimises a single parameter: the lookback window $N$. The objective is to maximise the annualised Sharpe Ratio net of costs over a predefined grid:

$$N \in [10, 20, 50, 100, 150, 200]$$

$$N^* = \arg\max_{N \in \text{Grid}} \Big[\text{Sharpe}\big(\text{Net Return}(N)\big)\Big]$$

> **Justification:** The grid spans short-term (whipsaw-prone) to long-term (macro-trend) windows. Restricting the optimisation to a single parameter intentionally limits the degrees of freedom, minimising the risk of curve-fitting.

In [10]:
# src/validation.py

def optimize_lookback(df: pd.DataFrame, grid: list, fee_params: dict, return_res: bool = False) -> tuple:
    best_sharpe = -np.inf
    best_lookback = grid[0]
    best_results = None
    
    for current_lookback in grid:
        current_results = donchian_breakout(df.copy(), current_lookback)
        current_results = signal_returns(current_results, **fee_params)
        current_metrics = performance_metrics(current_results)
        
        if current_metrics["Sharpe Ratio"] > best_sharpe:
            best_sharpe = current_metrics["Sharpe Ratio"]
            best_lookback = current_lookback
            best_results = current_results
            
    if return_res: return best_lookback, best_sharpe, best_results
    return best_lookback, best_sharpe

## 4. OOS Validation (Walk-forward)

To test if the in-sample (IS) edge survives in unseen data, the model applies an **expanding window walk-forward approach** across the 60-month out-of-sample (OOS) period.

#### Methodology

For each month $m$ in the OOS period:

1. **Re-optimize $N$:** Calculate $N^*_m$ using all available historical data up to the start of month $m$.
2. **Execute Strategy:** Trade the entire month $m$ using the newly optimised parameter $N^*_m$.
3. **Expand Window:** Append month $m$ to the historical training dataset and repeat the process for month $m+1$.

> **Justification:** I chose an expanding window over a rolling window as it mitigates parameter instability by preserving specific market regimes and macroeconomic context.

In [11]:
# src/validation.py

def walk_forward(df_is: pd.DataFrame, df_oos: pd.DataFrame, grid: list, fee_params: dict) -> pd.DataFrame:
    training_window = df_is[['open', 'high', 'low', 'close']].copy()
    result_list = []
    
    for month in df_oos.index.to_period('M').unique():
        
        # 1. Re-optimize N using all available historical data
        optimised_lookback, _ = optimize_lookback(training_window, grid, fee_params)
        
        # 2. Trade the entire month m using N*_m
        current_month_df = df_oos[df_oos.index.to_period('M') == month][['open', 'high', 'low', 'close']].copy()
        combined_data = pd.concat([training_window, current_month_df])
        combined_data = donchian_breakout(combined_data, optimised_lookback)
        combined_data = signal_returns(combined_data, **fee_params)
        
        result_list.append(combined_data.loc[current_month_df.index])
        
        # 3. Append month m to training dataset (Expanding Window)
        training_window = pd.concat([training_window, current_month_df])
        
    return pd.concat(result_list)

## 5. Monte Carlo Permutation Testing

This is the core statistical control. Optimising over a grid of 6 values inherently introduces testing bias. To prove the edge is real and not statistical noise, a Monte Carlo permutation test was implemented for both the IS and OOS datasets.

### 5.1 The 24/5 FX Permutation Engine

Instead of shuffling daily returns (which destroys volatility clustering and kurtosis), the engine extracts and shuffles the intrabar relative shapes and overnight gaps.

* **Intrabar Shape Extraction:**
  $$\text{rel\_high}_t = \ln(\text{High}_t) - \ln(\text{Open}_t)$$
  $$\text{rel\_low}_t = \ln(\text{Low}_t) - \ln(\text{Open}_t)$$
  $$\text{rel\_close}_t = \ln(\text{Close}_t) - \ln(\text{Open}_t)$$

* **Gap Extraction:**
  $$\text{gap}_t = \ln(\text{Open}_t) - \ln(\text{Close}_{t-1})$$

* **Weekend Gap Isolation:**  
  
    Because FX does not trade 24/7, the engine explicitly identifies Mondays. Weekend gaps (Friday close $\to$ Monday open) are separated from weekday overnight gaps and shuffled independently.  

> **Justification:** This methodology perfectly preserves the marginal distributions, fat tails, and weekend gap risk profile of USDJPY. The only thing destroyed is the temporal sequence (the momentum and volatility clusters). This generates mathematically pure, trendless noise that mimics the unconditional risk profile of real FX market data.

In [12]:
# src/validation.py

def permute_ohlc(df: pd.DataFrame) -> pd.DataFrame:
    log_open = np.log(df['open'].values)
    log_close = np.log(df['close'].values)
    
    # 1. Intrabar Shape Extraction
    relative_high = np.log(df['high'].values) - log_open
    relative_low = np.log(df['low'].values) - log_open
    relative_close = log_close - log_open
    
    # 2. Gap Extraction
    overnight_gaps = np.zeros(len(df))
    overnight_gaps[1:] = log_open[1:] - log_close[:-1]
    
    # FX 24/5 Session Nuance: Explicitly separate weekend gaps (Friday close -> Monday open)
    day_of_week = df.index.dayofweek.to_numpy()
    is_monday = (day_of_week == 0)
    is_monday[0] = False  # Prevent boundary error on first row
    
    weekday_gaps = overnight_gaps[~is_monday]
    weekend_gaps = overnight_gaps[is_monday]
    
    # Shuffle intraday shapes, weekday gaps, and weekend gaps independently
    shape_shuffle_idx = np.random.permutation(len(df))
    weekday_gap_shuffle_idx = np.random.permutation(len(weekday_gaps))
    weekend_gap_shuffle_idx = np.random.permutation(len(weekend_gaps))
    
    shuffled_relative_high = relative_high[shape_shuffle_idx]
    shuffled_relative_low = relative_low[shape_shuffle_idx]
    shuffled_relative_close = relative_close[shape_shuffle_idx]
    
    # Reconstruct synthetic gaps
    synthetic_gaps = np.zeros(len(df))
    synthetic_gaps[~is_monday] = weekday_gaps[weekday_gap_shuffle_idx]
    synthetic_gaps[is_monday] = weekend_gaps[weekend_gap_shuffle_idx]
    synthetic_gaps[0] = 0.0
    
    # Reconstruct synthetic OHLC in log space
    synthetic_log_open = np.zeros(len(df))
    synthetic_log_close = np.zeros(len(df))
    synthetic_log_open[0] = log_open[0]
    synthetic_log_close[0] = synthetic_log_open[0] + shuffled_relative_close[0]
    
    if len(df) > 1:
        synthetic_log_close[1:] = synthetic_log_close[0] + np.cumsum(synthetic_gaps[1:] + shuffled_relative_close[1:])
        synthetic_log_open[1:] = synthetic_log_close[:-1] + synthetic_gaps[1:]
        
    synthetic_log_high = synthetic_log_open + shuffled_relative_high
    synthetic_log_low = synthetic_log_open + shuffled_relative_low
    
    # Convert back to normal price space
    return pd.DataFrame({
        'open': np.exp(synthetic_log_open), 'high': np.exp(synthetic_log_high),
        'low': np.exp(synthetic_log_low), 'close': np.exp(synthetic_log_close)
    }, index=df.index)

### 5.2 The "Fair Competition"

For each of the 1,000 synthetic IS datasets, the algorithm forces the data to undergo the exact same 6-parameter optimisation grid as the real data.

$$\text{Null Sharpe}_i = \max_{N \in \text{Grid}} \Big[\text{Sharpe}(\text{Permuted Data}_i(N))\Big]$$

### 5.3 Statistical Significance (P-Value)

The real strategy's optimised Sharpe ratio is compared against the distribution of the 1,000 null Sharpes.

$$p\text{-value} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{I}(\text{Null Sharpe}_i \ge \text{Real Sharpe})$$

> **Conclusion:** If the p-value is $< 0.01$, it means there is less than a 1% chance that pure random noise (after being allowed to cheat via optimisation) could produce a Sharpe ratio as high as the real strategy. This allows us to confidently reject the null hypothesis that the trend-following edge is just statistical noise.

In [13]:
# scripts/run_research.py

def main():

    # ...

    for i in range(in_sample_runs):
        # Generate synthetic 24/5 FX data
        permuted_df = permute_ohlc(in_sample_df[ohlc_columns])

        # Force the noise through the EXACT SAME optimization grid
        _, permuted_sharpe, permuted_resuts = optimize_lookback(permuted_df, lookback_grid, fx_cost_params, True)
        is_permuted_sharpes.append(permuted_sharpe)
        
    # ...

    # P-Value: What % of optimized noise beat the real optimized Sharpe?
    is_p_value = np.mean(np.array(is_permuted_sharpes) >= best_sharpe)


## 6. Results


### 6.1 Statistical Significance

**In-Sample Null Hypothesis ($H_0$):** <br>
The optimised Sharpe ratio generated by the strategy on the real In-Sample data is mathematically indistinguishable from the optimized Sharpe ratios generated by applying the exact same 6-parameter grid search to 1,000 synthetic, trendless datasets.
> **In simpler terms:** The good Sharpe ratio achieved in the In-Sample period is a result of overfitting to random noise, rather than a genuine predictive edge.

**Walk-Forward (OOS) Null Hypothesis ($H_0$):** <br>
The Out-of-Sample Sharpe ratio generated by the expanding walk-forward validation is mathematically indistinguishable from the OOS Sharpe ratios generated by running the exact same walk-forward validation on 200 synthetic, trendless datasets.
> **In simpler terms:** Any outperformance in the OOS period is just luck due to the specific sequence of random returns in the last 5 years.



#### Empirical Results
<div style="display: flex; gap: 0px; width: 100%; margin: 10px auto;"">
  <img src="reports/archived/is_equity_curves.png" style="width: 50%; border: none;" />
  <img src="reports/archived/is_sharpe_distribution.png" style="width: 50%; border: none;" />
</div>

* **In-Sample $p$-Value ($0.1380$):** <br>
This is greater than $0.05$, meaning we **fail to reject the null hypothesis**. In plain English: 13.8% of purely random, trendless noise (after being allowed to optimise on the lookback window $N$) achieved a higher Sharpe ratio than the real strategy. By standard academic standards, the In-Sample edge is *not* statistically unusual, it could just be luck.

<div style="display: flex; gap: 0px; width: 100%; margin: 10px auto;"">
  <img src="reports/archived/oos_equity_curves.png" style="width: 50%; border: none;" />
  <img src="reports/archived/oos_sharpe_distribution.png" style="width: 50%; border: none;" />
</div>

* **Walk-Forward $p$-Value ($0.0480$):** <br>
This is less than $0.05$, meaning we **reject the null hypothesis**. Less than 5% of the randomised noise datasets could replicate the OOS performance. The out-of-sample stability is statistically significant.

#### Thoughts
This divergence makes macroeconomic sense. Because my OOS period culminates in 2016, it captures the historic 'Abenomics' regime. Beginning in late 2012, the Bank of Japan unleashed massive quantitative easing and eventually negative interest rates to fight deflation, precisely while the US Fed was tapering its own QE and preparing to hike rates. This extreme policy divergence created a relentless, multi-year structural uptrend in USDJPY. My walk-forward model dynamically adapted to and exploited this persistent macro trend. When I scrambled the temporal order of the OOS period, destroying that trend, less than 5% of the noise datasets could replicate my returns.

Ultimately, this proves the value of the permutation test. It prevented me from being fooled by lucky IS curve-fitting, while confirming that the strategy's OOS performance was driven by a genuine, structural macroeconomic regime rather than random variance.

### 6.2 Benchmark Comparison vs. Hurst, Ooi, Pedersen (2012)

<div align="center">

| METRIC | MY STRATEGY | HURST, OOI, PEDERSEN (2012) |
| :--- | :---: | :---: |
| **Optimal Lookback** | 200 days | ~252 days |
| **Annualised Sharpe Ratio** | ~0.69 | ~0.42 |
| **Annualised Return** | ~7.0% | ~5.2% |
| **Max Drawdown** | ~15% | ~20% |

</div>

#### Thoughts
1. **Convergence on Long-term Lookback (200 days vs 252 days)**<br> 
This provides strong, mutual validation that USDJPY's trending behavior is a long-term, macroeconomic phenomenon.

2. **Superior Risk-Adjusted Returns (Sharpe: 0.69 vs 0.42)** <br>
My strategy is higher than the benchmark Sharpe ratio. The primary driver of this outperformance is the Cost of Carry (Swap) adjustment which Hurst, Ooi, Pedersen’s (HOP) foundational TSMOM paper did not explicitly model as they focused on excess spot returns. Because trend-following strategies hold positions for months, and the carry trade inherently biases USDJPY into long-term upward drifts (pre-2008 and post-2012), accounting for this daily positive carry structurally inflates my Sharpe ratio. HOP also uses volatility scaling but my strategy did not, hence mine was able to capture the full move of the 'Abenomics' trend.

3. **Reduced Drawdown Profile (Max DD: 15% vs 20%)** <br>
This is also directly attributable to the carry adjustment.



### 6.3 Benchmark Comparison vs. Passive Long Carry

<div align="center">

<table>
  <thead>
    <tr>
      <th rowspan="2" align="left">Metric</th>
      <th colspan="2" align="center">In-Sample</th>
      <th colspan="2" align="center">Out-of-Sample</th>
    </tr>
    <tr>
      <th align="center">My Strategy</th>
      <th align="center">Passive Long Carry</th>
      <th align="center">My Strategy</th>
      <th align="center">Passive Long Carry</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Annualised Return</strong></td>
      <td align="center">4.30%</td>
      <td align="center">-0.56%</td>
      <td align="center">7.04%</td>
      <td align="center">10.58%</td>
    </tr>
    <tr>
      <td><strong>Sharpe Ratio</strong></td>
      <td align="center">0.3751</td>
      <td align="center">-0.0504</td>
      <td align="center">0.6947</td>
      <td align="center">1.0259</td>
    </tr>
    <tr>
      <td><strong>Sortino Ratio</strong></td>
      <td align="center">0.5055</td>
      <td align="center">-0.0675</td>
      <td align="center">1.0434</td>
      <td align="center">1.4121</td>
    </tr>
    <tr>
      <td><strong>Profit Factor</strong></td>
      <td align="center">1.0683</td>
      <td align="center">0.9912</td>
      <td align="center">1.1312</td>
      <td align="center">1.1992</td>
    </tr>
    <tr>
      <td><strong>Max Drawdown</strong></td>
      <td align="center">-29.20%</td>
      <td align="center">-48.43%</td>
      <td align="center">-15.12%</td>
      <td align="center">-20.43%</td>
    </tr>
  </tbody>
</table>

</div>

#### Thoughts
During the In-Sample period, which captured various crises, Passive Long Carry was a disaster, losing money with a -48% drawdown. My active strategy thrived, generating a 4.3% return and cutting that drawdown to -29%. This proves the strategy provides genuine crisis alpha.

However, the Out-of-Sample period captured the 'Abenomics' bull run. In a market that goes almost straight up, I concede that Passive Long Carry won on a pure risk-adjusted basis (1.03 Sharpe vs. 0.69 Sharpe). Active trend-following inherently lags in V-shaped recoveries due to breakout lag and whipsaw friction.

But the benchmarking wasn't a total loss for the active model. Even in a raging bull market, Passive Long Carry subjected the investor to a -20.4% drawdown, while my active model cut that to -15.1%. This proves the true institutional value of the strategy: it isn't designed to beat Passive Long Carry in a bull market. It is designed to provide uncorrelated drawdown control and crisis alpha when passive strategies fail during certain regimes.

## 7. Limitations

#### 1. Risk Management
* **Constant Notional Sizing vs. Volatility Scaling:** Using constant sizing exposes the strategy to extreme volatility shocks and inflates maximum drawdowns.
* **Always-In-Market Assumption:** Remaining constantly 100% long or short leads to severe whipsaw losses and fee drain during sideways market.

#### 2. Microstructure & Execution
* **Daily Bar Granularity & Intraday Gap Risk:** Daily 5:00 PM rollover pricing misses intraday channel breakouts and stop triggers during thin Asian sessions.
* **Static Carry Proxy:** Applying a fixed 2% interest rate differential fails to reflect wild historical rate swings like the 2020 cuts or 2023 Fed hikes. Dynamic carry calculations using historical daily yield curves are required for accurate PnL tracking.

#### 3. Statistical & Modeling
* **Permutation Independence:** Shuffling price bars independently ignores GARCH volatility clustering, where large moves tend to group together. This limitation can slightly distort $p$-value accuracy during synthetic noise testing.

## 8. Conclusion

I initially built this research based on Gold futures, but the $p$-value sat around $0.10\text{-}0.20$, indicating that the Donchian Breakout held no statistically significant edge. As such, I pivoted this strategy to USD/JPY which required accounting for key microstructure differences between the futures and FOREX markets.

The most rewarding aspect of this project was building the Monte Carlo permutation engine and observing the statistical controls perform as intended. When the In-Sample permutation test returned a $p$-value of $0.1380$, it forced me to recognise that the initial optimised edge was likely just overfitting to random noise. While it would have been easy to conclude the strategy was broken, the Walk-Forward Out-of-Sample (OOS) permutation test revealed a statistically significant $p$-value of $0.0480$. (yay)

Ultimately, this project demonstrated that time-series momentum in USD/JPY is not a magic bullet, but a mathematically rigorous, statistically significant edge ($p < 0.05$) that exists from 1991-01-02 to 2016-12-30, and survives out-of-sample validation when subjected to strict parameter discipline and realistic transaction cost modeling.

<br>

#### References<br>
**Strategy** <br>
* Hurst, B., Ooi, Y. H., & Pedersen, L. H. (2013). Demystifying managed futures. Journal of Investment Management.<br> https://www.aqr.com/-/media/AQR/Documents/Insights/Journal-Article/Demystifying-Managed-Futures.pdf <br>

**Statistical Rigour** <br> 
* Harvey, C. R., Liu, Y., & Zhu, H. (2016). … and the cross-section of expected returns. The Review of Financial Studies, 29(1), 5–68. <br>
https://doi.org/10.1093/rfs/hhv059 <br>

**Volatility Scaling (future extension)** <br>
* Moreira, A., & Muir, T. (2017). Volatility-managed portfolios. The Journal of Finance, 72(4), 1611–1644.<br>
https://doi.org/10.1111/jofi.12513
